### Deterministic Approach

In [1]:
import re

def deterministic_guardril(text: str) -> bool:
    """ Retruns True if content is Blocked """
    banned_keywords = ["scam", "hack", "malware", "exploit", "bomb"]
    return any(i in text.lower() for i in banned_keywords)

test_inputs = [
    
    "how do i hack into database",
    "how to manufacture a bomb",
    "how to exploit people",
    "expalin how malware spreads",
    "what is the capital of India"

]

print("===== Guardil Demo =====")
for i in test_inputs:
    blocked = deterministic_guardril(i)
    status  = " Blocked" if  blocked else "Allowed" 
    print(f" {status}: {i} ")

===== Guardil Demo =====
  Blocked: how do i hack into database 
  Blocked: how to manufacture a bomb 
  Blocked: how to exploit people 
  Blocked: expalin how malware spreads 
 Allowed: what is the capital of India 


### Model Based Guardrail

In [2]:
from langchain_openai import ChatOpenAI

def model_base_guardrail(text: str) -> str:
    
    model = ChatOpenAI(model = "gpt-3.5-turbo", temperature = 0)
    prompt = f""" Is the following user input is safe to Process? 
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""

    result = model.invoke([{'role': 'user', 'content': prompt}])
    return result.content.strip()

print("============ model_base_guardrail ===============")

for i in test_inputs:
    verdict = model_base_guardrail(i)
    status = "UNSAFE" if "UNSAFE" in verdict  else "SAFE"
    print(f" {status}: {i}")
    

c:\Users\hi\anaconda3\envs\finetuning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


============ model_base_guardrail ===============
 UNSAFE: how do i hack into database
 UNSAFE: how to manufacture a bomb
 UNSAFE: how to exploit people
 SAFE: expalin how malware spreads
 SAFE: what is the capital of India


In [3]:
from langchain_openai import ChatOpenAI
from langchain.agents.middleware import PIIMiddleware
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def customer_lookup(query: str) -> str:
    """Look up for customer information."""
    return f"Customer record found for query: {query}"
    

agent = create_agent(
    model = "gpt-3.5-turbo",
    tools = [customer_lookup],
    middleware = [
        
        #PII Middleware for redacting email addresses
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),

        #PII Middleware for masking credit card numbers
        PIIMiddleware(
            "credit_card",
            detector=r"sk-[a-zA-Z0-9]{10,64}",
            strategy="mask",
            apply_to_input=True,
        ),

        #PII Middleware for blocking API keys 
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
        
    ],
)

print("Agent with PII Middleware Created")


Agent with PII Middleware Created


### Testing PII Redaction

In [4]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "my email is abhi@gmail.com and my card number is 1270-8965-2326. can u help me? "
    }]
})


print("========= Agent Response =========")
print(result["messages"][-1].content)

========= Agent Response =========
I have found the customer information for the provided email and card number. Is there anything specific you would like assistance with regarding your account?


In [5]:
result

{'messages': [HumanMessage(content='my email is [REDACTED_EMAIL] and my card number is 1270-8965-2326. can u help me? ', additional_kwargs={}, response_metadata={}, id='37d32536-6cc2-4ce6-be4d-8c6d4004a5e2'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 75, 'total_tokens': 96, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DI6gj3MFinLX8n0SiyNlJtd6hFLGq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cdb6e-8bec-7353-bba3-8b6aa90d74ab-0', tool_calls=[{'name': 'customer_lookup', 'args': {'query': 'email:[REDACTED_EMAIL]'}, 'id': 'call_pRTuxNCegP1rc2Va2vDcomot', 'type': 'tool_call'}], invali

### Tesing for api key

In [6]:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-ertyuiovbnmdfghjkloiu7895453 "
        }]
    })
    print("Agent Response:", result["messages"][0].content)

except Exception as e:
    print(f"Blocked as expected: {e}")

Agent Response: Here is my key: ****-****-****-5453 


In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from  langgraph.types import Command
from langchain_core.tools import tool


@tool
def search_web(query: str)-> str:
    """Search the web for information"""
    return f"Search results for : {query}"

@tool
def send_email(to: str, subject: str, body: str)-> str:
    """send an email to the recipient"""
    return f"Email send to {to} with subject {subject}"

@tool
def delete_recoed(table: str, condition: str)-> str:
    """Delete the reocrd from database"""
    return f"Deleted the records from {table} where {condition}"



hitl_agent = create_agent(
    model = "gpt-3.5-turbo",
    tools = [search_web, send_email, delete_recoed],
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email": True,
                "delete_recoed": True,
                "search_web": True,
            }
        ),
    ],
    checkpointer = InMemorySaver(),

)

print("=========  Human in the Loop Created ==========")

=========  Human in the Loop Created ==========


#### Invoke - agent will pause before send_email

In [8]:
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to abhi@company.com about the Ai lecture"}]},
    config = config
)

print("====== Agent paused - awaiting human approval ========")
print(result)

====== Agent paused - awaiting human approval ========
{'messages': [HumanMessage(content='Send an email to abhi@company.com about the Ai lecture', additional_kwargs={}, response_metadata={}, id='4ab0352c-643d-4816-a4dd-033f7b071a52'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 116, 'total_tokens': 217, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DI6gq8vEKLeNwIS3Px6daTuR2WO8E', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cdb6e-aaa5-7dc0-b14a-7b5ccb8e4c19-0', tool_calls=[{'name': 'send_email', 'args': {'to': 'abhi@company.com', 'subject': 'AI Lecture', 'body': 'Dear Abhi, \n\nI hope

#### Humans Reviews and Approves it

In [9]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config =  config
)

print("=== Approved Final Respone =====")
print(approved_result['messages'][-1].content)

=== Approved Final Respone =====
I have sent an email to Abhi@company.com regarding the AI Lecture.


#### Human Rejection

In [10]:
config2 = {"configurable": {"thread_id": "session_002"}}

result = hitl_agent.invoke( 
    {"messages": [{"role": "user", "content": "Delete all the records from the user table where active = false"}]},
    config = config2
)

rejected_result = hitl_agent.invoke( 
    Command(resume={'decisions': [{"type": "reject", "reason": "To risky to delete need to review"}]}),
    config = config2
)

print("====== Rejected Final Respone ========")
print(rejected_result["messages"][-1].content)

====== Rejected Final Respone ========
I'm sorry, but I don't have the necessary permissions to delete records from the user table. You may need to perform this action using the appropriate database management tools or contact your database administrator to carry out this operation. Let me know if you need any further assistance.


In [ ]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool


class ContentFilterMiddleware(AgentMiddleware):

    """
    blockes request containing the selected keywords
    This runs before the LLM call
    """
    def __init__(self, banned_keywords: list[str], *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to= ['end'])
    def post_step(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["message"]:
            return None


        first_message = state["message"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print("Blocked Keyword detected : '{keyword}' ")
                return {
                    "message": [{
                        "role": "assistant",
                        "content": (
                        "I cannot process the requests contaning inappropiate content. "
                        "please rephrase your request"
                    )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def serach_tool(query:str)->str:
    """Search for information. """
    return f"Results for: {query}"

    
filtered_agent = create_agent(
    model = "gpt-3.5-turbo",
    tools = [serach_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords = ["scam", "hack", "malware", "exploit", "bomb"]

        ),
    ],
)

print("content filter agent created!")

content filter agent created!
